In [ ]:
### Sequence preparation before dada2 step

conda activate bioinfo

#Create a symbolic link so you don't touch your original data and only modify the symbolic


#fastqc reads to have an overview of the quality for the different methods (before the phred 30 filtering)
fastqc *_R1.fq.gz  -o fastqcR1_rawreads
fastqc *_R2.fq.gz  -o fastqcR2_rawreads

#Combine fastqc reports with multiqc

##Some R2 samples have a lot of polyA and polyG flagged as adapter content - check if after filtering they'll disappear

##Remove reads that have an AVERAGE phred score <30 - use batch_fastp_avgQ30_with_summary.sh
#make script executable:
chmod +x batch_fastp_avgQ30_with_summary.sh
#Run bash script in the directory with the fastq files:
./batch_fastp_avgQ30_with_summary.sh

#Filtering step removes between ~5 to ~7% of original reads - an acceptable threshold

##Do a quick check with fastqc to see how samples are now and then compare with the raw results from the sequencing
#Create fastqc_R1 and fastqc_R2 inside the fastp_filtered folder

fastqc *_R1.filtered.fq.gz  -o fastqc_R1
fastqc *_R2.filtered.fq.gz  -o fastqc_R2
#Then join all reports with multiqc

##Quality reports look much better! Reads with polyA and PolyG were removed!

#check if your sequences contain primers:

zgrep 'GTGCCAGCAGCCGCGGTAA' B7_R1.filtered.fq.gz | wc -l 

#cut primers with cutadapt 
# -g: fwd primer (5' direction); -G: rev primer 5' direction 
#  -m minimum read length, -M max read length


#do it for all samples
for sample in $(cat samples.txt)
do
    echo "On sample: $sample"
    cutadapt -g GTGYCAGCMGCCGCGGTAA \
             -G GGACTACNVGGGTWTCTAAT \
             -m 210 -M 230 --discard-untrimmed \
             -o ${sample}_R1_trimmed.fq.gz -p ${sample}_R2_trimmed.fq.gz \
             ${sample}_R1.filtered.fq.gz ${sample}_R2.filtered.fq.gz \
             >> cutadapt_primer_trimming_stats.txt 2>&1
done

#look at the output of the cutadapt stats file to get an idea of how things went - what fraction of reads were retained (col 2) and what fraction of bps were retained (col3):
paste samples.txt <(grep "passing" cutadapt_primer_trimming_stats.txt | cut -f3 -d "(" | tr -d ")") <(grep "filtered" cutadapt_primer_trimming_stats.txt | cut -f3 -d "(" | tr -d ")")

#Create a  directory for the fastqc of R1 and R2 reads
mkdir fastqc_R1_trimmed
mkdir fastqc_R2_trimmed

#Run fastqc to check quality of sequencing and send the output to the corresponding directories
fastqc *_R1_*  -o fastqc_R1_trimmed
fastqc *_R2_*  -o fastqc_R2_trimmed

#Change to the directory for fastqc and run multiqc to pool all the quality reports
cd fastqc_R1_trimmed
multiqc . # the dot means that multiqc is looking in the current directory

#Check the quality report
open -a firefox -g multiqc_report.html

#Samples are ready to be imported into R to run the dada2 pipeline